# Logic-Zero · Task 8: Full eval on 1530-puzzle set

Greedy-only eval (T=0) on the full eval set, per spec §5.5. Reports per-n accuracy + format compliance.

**Two modes** controlled by the cell below (`MODEL_KIND`):
- `'sft'` — load the SFT LoRA adapter from Drive
- `'base'` — untrained Qwen2.5-1.5B-Instruct baseline

**Run both in parallel:** open this notebook in 2 Colab tabs, set one to `'sft'` and one to `'base'`. Total wall time ~8-10h each (so 8-10h overall, not 16-20h).

**Output:**
- `results/eval_sft.json` or `results/eval_base.json` (local)
- Drive backup at `/MyDrive/logic-zero/checkpoints/sft/eval_{kind}.json`
- Progress file `eval_{kind}.inprogress.json` updated every 50 puzzles — safe to monitor mid-run

**Cost:** Colab compute only. No API calls.

## 0. Choose mode

**SET THIS** before running anything else.

In [ ]:
MODEL_KIND = 'sft'  # 'sft' or 'base'
assert MODEL_KIND in ('sft', 'base')
print(f'Running eval for: {MODEL_KIND}')

## 1. Verify GPU

In [ ]:
import torch, gc
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    free, total = torch.cuda.mem_get_info()
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {free/1e9:.1f}GB free / {total/1e9:.1f}GB total')
    assert free / 1e9 > 5, 'Less than 5GB free — restart runtime first'
else:
    raise RuntimeError('NO GPU — switch runtime type to GPU')

## 2. Pull repo + install deps

In [ ]:
import os
if not os.path.isdir('/content/logic-zero'):
    !git clone https://github.com/bsdnn/logic-zero.git /content/logic-zero
%cd /content/logic-zero
!git pull origin main

In [ ]:
import subprocess
def pip_inst(specs, retries=3):
    for i in range(retries):
        r = subprocess.run(['pip', 'install', '-q', '--default-timeout=180'] + specs,
                           capture_output=True, text=True)
        if r.returncode == 0:
            print(f'✓ {specs}')
            return
        print(f'[retry {i+1}/{retries}] {r.stderr[-300:]}')
    raise RuntimeError(f'pip install failed: {specs}')
pip_inst(['trl>=0.13.0,<0.15.0', 'peft>=0.13.0,<0.16.0', 'datasets>=3.0.0,<4.0.0', 'accelerate>=1.0.0'])
pip_inst(['z3-solver', 'python-dotenv'])  # train.common imports z3
import transformers, peft, z3
print(f'transformers={transformers.__version__}  peft={peft.__version__}  z3={z3.get_version_string()}')

## 3. Mount Drive + (if SFT) restore checkpoint

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import shutil
local_ckpt = '/content/logic-zero/results/checkpoints/sft/best'
drive_ckpt = '/content/drive/MyDrive/logic-zero/checkpoints/sft/best'
if MODEL_KIND == 'sft':
    if not os.path.isdir(local_ckpt):
        assert os.path.isdir(drive_ckpt), f'No checkpoint in Drive: {drive_ckpt}'
        os.makedirs(os.path.dirname(local_ckpt), exist_ok=True)
        shutil.copytree(drive_ckpt, local_ckpt)
        print(f'✓ Restored {drive_ckpt} → {local_ckpt}')
    else:
        print(f'✓ Already present: {local_ckpt}')
    adapter_size = os.path.getsize(f'{local_ckpt}/adapter_model.safetensors') / 1e6
    assert adapter_size > 10, f'Adapter too small ({adapter_size}MB)'
    print(f'  Adapter: {adapter_size:.1f}MB')
else:
    print('Base model — no adapter to restore.')

## 4. Run eval

Greedy-only (T=0). 1530 puzzles. ~8-10h on T4 (faster on L4).

Progress prints every 25 puzzles + ETA. Resume snapshot saved to `eval_{kind}.inprogress.json` every 50 puzzles — if Colab disconnects, you can read that file to see how far it got.

**Tip:** before running, verify Drive is mounted by checking the cell above succeeded. If the cell completes and Drive auto-unmounts mid-run, the final backup step will fail but the local result is still saved.

In [ ]:
import time
out_name = f'eval_{MODEL_KIND}.json'
out_path = f'results/{out_name}'
adapter_arg = '--adapter results/checkpoints/sft/best' if MODEL_KIND == 'sft' else ''
print(f'Output: {out_path}')
print(f'Started at: {time.strftime("%Y-%m-%d %H:%M:%S")}\n')
!python -u -m eval.run_eval {adapter_arg} --out {out_path} 2>&1 | tee logs/eval_{MODEL_KIND}.log

## 5. Verify + backup to Drive

In [ ]:
import json
result = json.load(open(out_path))
print(f'Model: {result["model"]}')
print(f'Puzzles: {result["n_puzzles"]}')
print(f'\n=== Greedy results ===')
g = result['greedy']
print(f'Overall acc: {g["overall_acc"]:.3f} ({g["total_correct"]}/{g["total"]})')
print(f'Format compliance: {g["format_compliance"]:.3f}')
print(f'Duration: {g["duration_sec"]/3600:.2f}h')
print(f'\nPer-n accuracy:')
for n in sorted(g['per_bucket_acc'], key=int):
    c = g['per_bucket_correct'][n] if n in g['per_bucket_correct'] else g['per_bucket_correct'][int(n)]
    t = g['per_bucket_total'][n] if n in g['per_bucket_total'] else g['per_bucket_total'][int(n)]
    a = g['per_bucket_acc'][n]
    bar = '█' * int(a * 30)
    print(f'  n={n}: {c:3d}/{t} = {a:.1%}  {bar}')

# Backup to Drive
drive_out = f'/content/drive/MyDrive/logic-zero/checkpoints/sft/{out_name}'
shutil.copy(out_path, drive_out)
shutil.copy(f'logs/eval_{MODEL_KIND}.log', f'/content/drive/MyDrive/logic-zero/checkpoints/sft/eval_{MODEL_KIND}.log')
print(f'\n✓ Backed up to {drive_out}')

## ✅ Task 8 done

**Output:** `results/eval_{kind}.json` (local + Drive backup)

### Next steps

1. **If you only ran one mode**, open another Colab tab with this same notebook, change `MODEL_KIND`, and run again. They're independent.
2. **Once both finish**, compare in `eval/compare.py` (Task 20) — but typically just look at the SFT vs base per-n table side by side.
3. **Sanity gate (Task 10)**:
   - SFT overall acc ≥ 30% → ✅ pipeline healthy, proceed to **Task 9 DPO**
   - SFT overall acc < 30% but ≥ 15% → cautiously proceed, expect DPO to lift most on n=2-4
   - SFT overall acc < 15% → debug before DPO

### Reading per-n carefully

Pay attention to **n=7**: we trained on n=2-6 only. n=7 tests out-of-distribution generalization. Expect base model ~0%, SFT model 5-15% (the model learned the *method*, may transfer partially).